# Conventional AI versus UVIF-Guided Pneumonia Decision Support Notebook

This upgraded Colab notebook provides a concrete computational illustration of the philosophical transition from **conventional prediction-centric AI** to **UVIF-guided decision-centric intelligence**.

The workflow produces two explicit result streams:

1. **Conventional AI:** forced classification using the best-performing predictive model at the default threshold.
2. **UVIF-oriented AI:** decision-support interpretation using uncertainty, calibration, risk, threshold control, and selective clinical review.

The final artifacts include side-by-side tables and figures comparing conventional AI on the left and UVIF on the right.


In [ ]:
# ============================================================
# 1. Mount Google Drive and prepare reproducible output folders
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, warnings
warnings.filterwarnings("ignore")

PROJECT_NAME = "UVIF_Diagnostics_PneumoniaMNIST"
BASE_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME
FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
MODEL_DIR = BASE_DIR / "models"
OTHER_DIR = BASE_DIR / "others"

for d in [FIG_DIR, TABLE_DIR, MODEL_DIR, OTHER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY = BASE_DIR / "outputs_summary.txt"
OUTPUT_SUMMARY.write_text("UVIF-Guided Diagnostics-Oriented PneumoniaMNIST Pipeline\n", encoding="utf-8")

def log(msg):
    print(msg)
    with open(OUTPUT_SUMMARY, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

log(f"[INFO] Output directory: {BASE_DIR}")

In [ ]:
# ============================================================
# 2. Install required packages
# ============================================================
!pip install -q medmnist xgboost lightgbm shap lime scikit-learn joblib

In [ ]:
# ============================================================
# 3. Imports and reproducibility setup
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, log_loss,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

log(f"[INFO] TensorFlow version: {tf.__version__}")

## Dataset path

The notebook first searches common Drive locations. If your file is elsewhere, update `DATA_PATH` manually.

In [ ]:
# ============================================================
# 4. Locate and load PneumoniaMNIST .npz directly
# ============================================================
candidate_paths = [
    "/content/drive/MyDrive/Datasets/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Pneumonia/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Medical/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/MedMNIST/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Heart/pneumoniamnist.npz",
]

DATA_PATH = None
for p in candidate_paths:
    if Path(p).exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    # Fallback: search under MyDrive/Datasets
    matches = list(Path("/content/drive/MyDrive/Datasets").rglob("pneumoniamnist.npz"))
    if matches:
        DATA_PATH = str(matches[0])

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find pneumoniamnist.npz. Please place it under /content/drive/MyDrive/Datasets/ "
        "or update DATA_PATH manually."
    )

log(f"[PATH CHECK] Dataset found: {DATA_PATH}")

data = np.load(DATA_PATH)
log(f"[INFO] Keys in .npz file: {data.files}")

X_train_raw = data["train_images"]
y_train_raw = data["train_labels"]
X_val_raw   = data["val_images"]
y_val_raw   = data["val_labels"]
X_test_raw  = data["test_images"]
y_test_raw  = data["test_labels"]

log(f"[INFO] Train images: {X_train_raw.shape}, labels: {y_train_raw.shape}")
log(f"[INFO] Val images:   {X_val_raw.shape}, labels: {y_val_raw.shape}")
log(f"[INFO] Test images:  {X_test_raw.shape}, labels: {y_test_raw.shape}")

In [ ]:
# ============================================================
# 5. Preprocessing
# ============================================================
def preprocess_images(X):
    X = X.astype("float32") / 255.0
    if X.ndim == 3:
        X = X[..., np.newaxis]
    return X

X_train = preprocess_images(X_train_raw)
X_val   = preprocess_images(X_val_raw)
X_test  = preprocess_images(X_test_raw)

y_train = y_train_raw.reshape(-1).astype(int)
y_val   = y_val_raw.reshape(-1).astype(int)
y_test  = y_test_raw.reshape(-1).astype(int)

CLASS_NAMES = ["Normal", "Pneumonia"]

log(f"[INFO] X_train processed: {X_train.shape}")
log(f"[INFO] X_val processed:   {X_val.shape}")
log(f"[INFO] X_test processed:  {X_test.shape}")

class_counts = pd.DataFrame({
    "split": ["train"] * len(y_train) + ["val"] * len(y_val) + ["test"] * len(y_test),
    "label": np.concatenate([y_train, y_val, y_test])
})
dist = class_counts.groupby(["split", "label"]).size().reset_index(name="count")
dist["class_name"] = dist["label"].map({0: "Normal", 1: "Pneumonia"})
display(dist)
dist.to_csv(TABLE_DIR / "table_class_distribution.csv", index=False)
log("\n[CLASS DISTRIBUTION]")
log(dist.to_string(index=False))

In [ ]:
# ============================================================
# 6. Visual inspection of representative images
# ============================================================
plt.figure(figsize=(8, 4))
plot_idx = 1
for cls in [0, 1]:
    indices = np.where(y_train == cls)[0][:5]
    for idx in indices:
        plt.subplot(2, 5, plot_idx)
        plt.imshow(X_train[idx].squeeze(), cmap="gray")
        plt.title(CLASS_NAMES[cls])
        plt.axis("off")
        plot_idx += 1
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_representative_pneumoniamnist_images.png", dpi=300, bbox_inches="tight")
plt.show()

## Compact CNN

The model is intentionally lightweight because PneumoniaMNIST images are 28×28 grayscale images. This keeps the workflow reproducible and suitable for Colab.

In [ ]:
# ============================================================
# 7. Build compact CNN model
# ============================================================
def build_compact_cnn(input_shape=(28, 28, 1)):
    inputs = layers.Input(shape=input_shape, name="input_image")

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="last_conv")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D(name="embedding")(x)

    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="risk_output")(x)

    model = models.Model(inputs, outputs, name="Compact_CNN_PneumoniaMNIST")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

cnn_model = build_compact_cnn(input_shape=X_train.shape[1:])
cnn_model.summary(print_fn=log)

In [ ]:
# ============================================================
# 8. Train CNN
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
log(f"[INFO] Class weights: {class_weight}")

ckpt_path = MODEL_DIR / "best_compact_cnn.keras"

cb = [
    callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=1)
]

history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight,
    callbacks=cb,
    verbose=1
)

hist_df = pd.DataFrame(history.history)
hist_df.to_csv(TABLE_DIR / "table_cnn_training_history.csv", index=False)
display(hist_df.tail())

In [ ]:
# ============================================================
# 9. Plot CNN training curves
# ============================================================
plt.figure()
plt.plot(hist_df["loss"], label="Training loss")
plt.plot(hist_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Diagnostics - CNN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_cnn_loss_curve.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(hist_df["accuracy"], label="Training accuracy")
plt.plot(hist_df["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Diagnostics - CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_cnn_accuracy_curve.png", dpi=300, bbox_inches="tight")
plt.show()

if "auc" in hist_df.columns:
    plt.figure()
    plt.plot(hist_df["auc"], label="Training AUC")
    plt.plot(hist_df["val_auc"], label="Validation AUC")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.title("Diagnostics - CNN Training and Validation AUC")
    plt.legend()
    plt.grid(True)
    plt.savefig(FIG_DIR / "figure_cnn_auc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# 10. Evaluation utilities
# ============================================================
def expected_calibration_error_binary(y_true, prob_pos, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    pred = (prob_pos >= 0.5).astype(int)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (prob_pos >= lo) & (prob_pos <= hi)
        else:
            mask = (prob_pos >= lo) & (prob_pos < hi)

        if np.any(mask):
            acc = np.mean(y_true[mask] == pred[mask])
            conf = np.mean(prob_pos[mask])
            weight = np.mean(mask)
            ece += np.abs(acc - conf) * weight
            rows.append({
                "bin_low": lo,
                "bin_high": hi,
                "count": int(np.sum(mask)),
                "accuracy": acc,
                "confidence": conf
            })

    return float(ece), pd.DataFrame(rows)

def evaluate_binary_model(name, y_true, prob_pos):
    prob_pos = np.asarray(prob_pos).reshape(-1)
    pred = (prob_pos >= 0.5).astype(int)

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1-score": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob_pos),
        "Log Loss": log_loss(y_true, np.column_stack([1 - prob_pos, prob_pos])),
        "ECE": expected_calibration_error_binary(y_true, prob_pos)[0]
    }

In [ ]:
# ============================================================
# 11. Evaluate CNN on test set
# ============================================================
cnn_prob = cnn_model.predict(X_test, verbose=0).reshape(-1)
cnn_pred = (cnn_prob >= 0.5).astype(int)

cnn_result = evaluate_binary_model("Compact CNN", y_test, cnn_prob)
cnn_results_df = pd.DataFrame([cnn_result])
display(cnn_results_df)
cnn_results_df.to_csv(TABLE_DIR / "table_cnn_test_performance.csv", index=False)

log("\n[CNN TEST PERFORMANCE]")
log(cnn_results_df.to_string(index=False))

cm = confusion_matrix(y_test, cnn_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title("Confusion Matrix - Compact CNN")
plt.savefig(FIG_DIR / "figure_confusion_matrix_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

report_df = pd.DataFrame(classification_report(y_test, cnn_pred, target_names=CLASS_NAMES, output_dict=True)).T
display(report_df)
report_df.to_csv(TABLE_DIR / "table_classification_report_cnn.csv")

In [ ]:
# ============================================================
# 12. ROC curve and calibration curve for CNN
# ============================================================
RocCurveDisplay.from_predictions(y_test, cnn_prob)
plt.title("Diagnostics ROC Curve - Compact CNN")
plt.savefig(FIG_DIR / "figure_roc_curve_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

prob_true, prob_pred = calibration_curve(y_test, cnn_prob, n_bins=10)
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label="CNN")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Diagnostics Calibration Curve - Compact CNN")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_calibration_curve_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

ece, calib_table = expected_calibration_error_binary(y_test, cnn_prob, n_bins=10)
display(calib_table)
calib_table.to_csv(TABLE_DIR / "table_calibration_bins_cnn.csv", index=False)
log(f"[CNN] Expected Calibration Error: {ece:.5f}")

## CNN embeddings + machine-learning ensemble

The CNN is used as a feature extractor. Its embedding layer produces compact representations that are passed to classical ML models and a soft-voting ensemble.

In [ ]:
# ============================================================
# 13. Extract CNN embeddings
# ============================================================
embedding_model = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("embedding").output
)

Z_train = embedding_model.predict(X_train, verbose=0)
Z_val   = embedding_model.predict(X_val, verbose=0)
Z_test  = embedding_model.predict(X_test, verbose=0)

# Use train + validation for ML training after CNN selection
Z_train_full = np.vstack([Z_train, Z_val])
y_train_full = np.concatenate([y_train, y_val])

log(f"[INFO] Embedding shapes: train={Z_train.shape}, val={Z_val.shape}, test={Z_test.shape}")
np.save(OTHER_DIR / "Z_train_full_embeddings.npy", Z_train_full)
np.save(OTHER_DIR / "Z_test_embeddings.npy", Z_test)

In [ ]:
# ============================================================
# 14. Train classical ML models on CNN embeddings
# ============================================================
scale_pos_weight = (len(y_train_full) - y_train_full.sum()) / max(y_train_full.sum(), 1)
log(f"[INFO] scale_pos_weight for XGBoost: {scale_pos_weight:.3f}")

ml_models = {
    "LogisticRegression Embeddings": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    ]),
    "RandomForest Embeddings": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "ExtraTrees Embeddings": ExtraTreesClassifier(
        n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "XGBoost Embeddings": XGBClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=3,
        subsample=0.85, colsample_bytree=0.85,
        eval_metric="logloss", random_state=SEED,
        scale_pos_weight=float(scale_pos_weight)
    ),
    "LightGBM Embeddings": LGBMClassifier(
        n_estimators=400, learning_rate=0.03,
        class_weight="balanced", random_state=SEED
    )
}

ml_results = []
ml_probabilities = {}
ml_predictions = {}
trained_ml_models = {}

for name, clf in ml_models.items():
    log(f"[TRAIN] {name}")
    clf.fit(Z_train_full, y_train_full)
    prob = clf.predict_proba(Z_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    ml_results.append(evaluate_binary_model(name, y_test, prob))
    ml_probabilities[name] = prob
    ml_predictions[name] = pred
    trained_ml_models[name] = clf

ml_results_df = pd.DataFrame(ml_results).sort_values("F1-score", ascending=False)
display(ml_results_df)
ml_results_df.to_csv(TABLE_DIR / "table_embedding_ml_model_performance.csv", index=False)
log("\n[EMBEDDING ML PERFORMANCE]")
log(ml_results_df.to_string(index=False))

In [ ]:
# ============================================================
# 15. Soft-voting ensemble on embeddings
# ============================================================
soft_voting = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1)),
        ("et", ExtraTreesClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1)),
        ("xgb", XGBClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=3,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric="logloss", random_state=SEED,
            scale_pos_weight=float(scale_pos_weight)
        )),
        ("lgbm", LGBMClassifier(n_estimators=400, learning_rate=0.03, class_weight="balanced", random_state=SEED))
    ],
    voting="soft"
)

log("[TRAIN] SoftVoting Embeddings")
soft_voting.fit(Z_train_full, y_train_full)

sv_prob = soft_voting.predict_proba(Z_test)[:, 1]
sv_pred = (sv_prob >= 0.5).astype(int)
sv_result = evaluate_binary_model("SoftVoting Embeddings", y_test, sv_prob)

all_results_df = pd.concat([
    cnn_results_df,
    ml_results_df,
    pd.DataFrame([sv_result])
], ignore_index=True).sort_values("F1-score", ascending=False)

display(all_results_df)
all_results_df.to_csv(TABLE_DIR / "table_all_model_performance.csv", index=False)
log("\n[ALL MODEL PERFORMANCE]")
log(all_results_df.to_string(index=False))

In [ ]:
# ============================================================
# 16. Select best model and save performance artifacts
# ============================================================
best_model_name = all_results_df.iloc[0]["Model"]
log(f"[BEST MODEL] {best_model_name}")

if best_model_name == "Compact CNN":
    best_prob = cnn_prob
    best_pred = cnn_pred
elif best_model_name == "SoftVoting Embeddings":
    best_prob = sv_prob
    best_pred = sv_pred
else:
    best_prob = ml_probabilities[best_model_name]
    best_pred = ml_predictions[best_model_name]

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title(f"Diagnostics Confusion Matrix - {best_model_name}")
plt.savefig(FIG_DIR / "figure_confusion_matrix_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title(f"Diagnostics ROC Curve - {best_model_name}")
plt.savefig(FIG_DIR / "figure_roc_curve_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

prob_true, prob_pred = calibration_curve(y_test, best_prob, n_bins=10)
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label=best_model_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title(f"Diagnostics Calibration Curve - {best_model_name}")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_calibration_curve_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

best_report_df = pd.DataFrame(classification_report(y_test, best_pred, target_names=CLASS_NAMES, output_dict=True)).T
display(best_report_df)
best_report_df.to_csv(TABLE_DIR / "table_classification_report_best_model.csv")

## UVIF-guided diagnostics layer

This section keeps UVIF restrained and practical. It does not replace the trained CNN or embedding classifiers. Instead, it adds a decision-reliability layer that combines predictive performance, calibration reliability, false-negative risk, specificity burden, model complexity, and explainability utility into a transparent model-selection and threshold-control score.

The purpose is to make the notebook defensible for a Diagnostics-oriented manuscript: the model is judged not only by AUC or F1-score, but also by calibration, threshold behavior, screening sensitivity, and decision risk.


In [ ]:
# ============================================================
# 16B. UVIF-guided diagnostics layer for model selection
# ============================================================
# UVIF is used here as a restrained decision-reliability layer.
# It does not claim a new classifier architecture; it provides a principled
# way to combine discrimination, calibration, sensitivity, specificity,
# false-negative risk, complexity, and explainability utility.

MODEL_PROBABILITIES = {"Compact CNN": cnn_prob, "SoftVoting Embeddings": sv_prob}
MODEL_PROBABILITIES.update(ml_probabilities)

# Practical complexity priors in [0, 1]. These are intentionally simple and transparent.
# Lower values indicate more practical / lightweight use in a Diagnostics workflow.
complexity_prior = {
    "LogisticRegression Embeddings": 0.20,
    "RandomForest Embeddings": 0.65,
    "ExtraTrees Embeddings": 0.65,
    "XGBoost Embeddings": 0.70,
    "LightGBM Embeddings": 0.70,
    "SoftVoting Embeddings": 0.90,
    "Compact CNN": 0.55,
}

# XAI utility priors in [0, 1]. CNN supports Grad-CAM/LIME; embedding models support SHAP.
# Hybrid/ensemble models receive high values because they can be interpreted through multiple mechanisms.
xai_utility_prior = {
    "LogisticRegression Embeddings": 0.70,
    "RandomForest Embeddings": 0.80,
    "ExtraTrees Embeddings": 0.80,
    "XGBoost Embeddings": 0.80,
    "LightGBM Embeddings": 0.85,
    "SoftVoting Embeddings": 0.75,
    "Compact CNN": 0.85,
}

UVIF_WEIGHTS = {
    "auc": 0.20,
    "sensitivity": 0.25,
    "specificity": 0.15,
    "f1": 0.15,
    "calibration_reliability": 0.10,
    "xai_utility": 0.10,
    "false_negative_penalty": 0.15,
    "complexity_penalty": 0.05,
}

uvif_rows = []
for model_name, prob in MODEL_PROBABILITIES.items():
    pred = (np.asarray(prob) >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    f1 = f1_score(y_test, pred, zero_division=0)
    auc = roc_auc_score(y_test, prob)
    ece = expected_calibration_error_binary(y_test, prob, n_bins=10)[0]
    fn_rate = fn / max(tp + fn, 1)
    fp_rate = fp / max(tn + fp, 1)
    calibration_reliability = max(0.0, 1.0 - ece)
    complexity = complexity_prior.get(model_name, 0.75)
    xai_utility = xai_utility_prior.get(model_name, 0.70)

    uvif_score = (
        UVIF_WEIGHTS["auc"] * auc
        + UVIF_WEIGHTS["sensitivity"] * sensitivity
        + UVIF_WEIGHTS["specificity"] * specificity
        + UVIF_WEIGHTS["f1"] * f1
        + UVIF_WEIGHTS["calibration_reliability"] * calibration_reliability
        + UVIF_WEIGHTS["xai_utility"] * xai_utility
        - UVIF_WEIGHTS["false_negative_penalty"] * fn_rate
        - UVIF_WEIGHTS["complexity_penalty"] * complexity
    )

    uvif_rows.append({
        "Model": model_name,
        "UVIF Score": uvif_score,
        "ROC-AUC": auc,
        "Sensitivity / Pneumonia Recall": sensitivity,
        "Specificity / Normal Recall": specificity,
        "Precision": precision,
        "F1-score": f1,
        "ECE": ece,
        "Calibration Reliability (1-ECE)": calibration_reliability,
        "False Negatives": int(fn),
        "False Positives": int(fp),
        "FN Rate": fn_rate,
        "FP Rate": fp_rate,
        "XAI Utility Prior": xai_utility,
        "Complexity Prior": complexity,
    })

uvif_model_selection_df = pd.DataFrame(uvif_rows).sort_values("UVIF Score", ascending=False)
display(uvif_model_selection_df)
uvif_model_selection_df.to_csv(TABLE_DIR / "table_uvif_diagnostics_model_selection.csv", index=False)

uvif_model_name = uvif_model_selection_df.iloc[0]["Model"]
uvif_prob = MODEL_PROBABILITIES[uvif_model_name]
uvif_pred_050 = (uvif_prob >= 0.5).astype(int)
log("\n[UVIF DIAGNOSTICS MODEL SELECTION]")
log(uvif_model_selection_df.to_string(index=False))
log(f"[UVIF SELECTED MODEL] {uvif_model_name}")

# Diagnostics plot: UVIF model-selection ranking
plt.figure(figsize=(9, 4.5))
plot_df = uvif_model_selection_df.sort_values("UVIF Score", ascending=True)
plt.barh(plot_df["Model"], plot_df["UVIF Score"])
plt.xlabel("UVIF diagnostics score")
plt.title("Diagnostics - UVIF-Guided Model Selection Score")
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_diagnostics_uvif_model_selection_score.png", dpi=300, bbox_inches="tight")
plt.show()


## UVIF-guided threshold control

This section searches decision thresholds and selects an operating point that remains screening-oriented while penalizing excessive false positives, false negatives, and calibration unreliability. The generated curves explicitly use Diagnostics-oriented labels and can be used directly in the manuscript.


In [ ]:
# ============================================================
# 16C. UVIF-guided threshold control and Diagnostics curves
# ============================================================
thresholds = np.linspace(0.01, 0.99, 99)
threshold_rows = []
base_ece = expected_calibration_error_binary(y_test, uvif_prob, n_bins=10)[0]

for tau in thresholds:
    pred_tau = (uvif_prob >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred_tau).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    f1 = f1_score(y_test, pred_tau, zero_division=0)
    balanced_acc = balanced_accuracy_score(y_test, pred_tau)
    fn_rate = fn / max(tp + fn, 1)
    fp_rate = fp / max(tn + fp, 1)

    # Screening-oriented UVIF utility: sensitivity is dominant, but false positives,
    # calibration error, and excessive threshold imbalance remain penalized.
    uvif_threshold_utility = (
        0.35 * sensitivity
        + 0.20 * specificity
        + 0.15 * precision
        + 0.15 * f1
        + 0.10 * balanced_acc
        - 0.25 * fn_rate
        - 0.10 * fp_rate
        - 0.10 * base_ece
    )

    threshold_rows.append({
        "threshold": tau,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1,
        "balanced_accuracy": balanced_acc,
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "fn_rate": fn_rate,
        "fp_rate": fp_rate,
        "ece_at_model_level": base_ece,
        "uvif_threshold_utility": uvif_threshold_utility,
    })

uvif_threshold_df = pd.DataFrame(threshold_rows)
uvif_threshold_df.to_csv(TABLE_DIR / "table_uvif_diagnostics_threshold_control.csv", index=False)

uvif_tau_star = float(uvif_threshold_df.loc[uvif_threshold_df["uvif_threshold_utility"].idxmax(), "threshold"])
uvif_pred = (uvif_prob >= uvif_tau_star).astype(int)
uvif_threshold_summary = uvif_threshold_df.loc[uvif_threshold_df["threshold"].sub(uvif_tau_star).abs().idxmin()].to_frame().T

display(uvif_threshold_summary)
uvif_threshold_summary.to_csv(TABLE_DIR / "table_uvif_selected_threshold_summary.csv", index=False)
log(f"[UVIF SELECTED THRESHOLD] tau* = {uvif_tau_star:.2f} for {uvif_model_name}")
log(uvif_threshold_summary.to_string(index=False))

# Diagnostics curve 1: UVIF utility by threshold
plt.figure(figsize=(8, 4.5))
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["uvif_threshold_utility"], linewidth=2)
plt.axvline(uvif_tau_star, linestyle="--", label=f"UVIF threshold = {uvif_tau_star:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("UVIF threshold utility")
plt.title(f"Diagnostics - UVIF Threshold Utility Curve ({uvif_model_name})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_diagnostics_uvif_threshold_utility_curve.png", dpi=300, bbox_inches="tight")
plt.show()

# Diagnostics curve 2: sensitivity-specificity trade-off
plt.figure(figsize=(8, 4.5))
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["sensitivity"], label="Sensitivity / pneumonia recall")
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["specificity"], label="Specificity / normal recall")
plt.axvline(uvif_tau_star, linestyle="--", label=f"UVIF threshold = {uvif_tau_star:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Diagnostic operating metric")
plt.title(f"Diagnostics - Sensitivity-Specificity Threshold Profile ({uvif_model_name})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_diagnostics_sensitivity_specificity_threshold_profile.png", dpi=300, bbox_inches="tight")
plt.show()

# Diagnostics curve 3: precision-recall-threshold behavior
plt.figure(figsize=(8, 4.5))
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["precision"], label="Precision")
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["sensitivity"], label="Recall / sensitivity")
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["f1_score"], label="F1-score")
plt.axvline(uvif_tau_star, linestyle="--", label=f"UVIF threshold = {uvif_tau_star:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title(f"Diagnostics - Precision-Recall-F1 Threshold Profile ({uvif_model_name})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_diagnostics_precision_recall_f1_threshold_profile.png", dpi=300, bbox_inches="tight")
plt.show()

# Diagnostics curve 4: false-positive / false-negative burden
plt.figure(figsize=(8, 4.5))
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["false_negatives"], label="False negatives")
plt.plot(uvif_threshold_df["threshold"], uvif_threshold_df["false_positives"], label="False positives")
plt.axvline(uvif_tau_star, linestyle="--", label=f"UVIF threshold = {uvif_tau_star:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Case count")
plt.title(f"Diagnostics - Error Burden Across Thresholds ({uvif_model_name})")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_diagnostics_error_burden_threshold_profile.png", dpi=300, bbox_inches="tight")
plt.show()

# Confusion matrix at UVIF-selected threshold
cm_uvif = confusion_matrix(y_test, uvif_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_uvif, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title(f"Diagnostics Confusion Matrix - UVIF Threshold ({uvif_model_name}, tau={uvif_tau_star:.2f})")
plt.savefig(FIG_DIR / "figure_diagnostics_confusion_matrix_uvif_threshold.png", dpi=300, bbox_inches="tight")
plt.show()

uvif_report_df = pd.DataFrame(classification_report(y_test, uvif_pred, target_names=CLASS_NAMES, output_dict=True)).T
display(uvif_report_df)
uvif_report_df.to_csv(TABLE_DIR / "table_diagnostics_uvif_threshold_classification_report.csv")


## Manuscript-ready comparison: Conventional AI versus UVIF

This section is the main comparative evidence block for the article. It keeps the trained models unchanged and contrasts two decision philosophies:

- **Conventional AI:** minimizes prediction error and forces every case into a class decision using a default threshold of 0.50.
- **UVIF:** balances prediction, uncertainty, risk, clinical burden, and safe action using a conservative operational threshold of 0.98 and an explicit review pathway.

The section generates exactly the requested manuscript artifacts:

- **Table 1:** quantitative Conventional AI versus UVIF comparison.
- **Table 2:** conceptual and decision comparison.
- **Figure 1:** performance bars.
- **Figure 2:** side-by-side confusion matrices.
- **Figure 3:** false-negative / false-positive burden comparison.
- **Figure 4:** UVIF decision-equilibrium figure.


In [ ]:
# ============================================================
# 16D. Manuscript tables: Conventional AI versus UVIF
# ============================================================
# This section is intentionally decision-focused rather than model-complexity-focused.
# Conventional AI uses the best predictive model at tau = 0.50.
# UVIF uses the UVIF-selected model probabilities but applies a conservative
# manuscript decision threshold tau = 0.98 to emphasize operational equilibrium:
# prediction + uncertainty + risk + clinical burden + safe action.

from sklearn.metrics import matthews_corrcoef

CONVENTIONAL_THRESHOLD = 0.50
UVIF_MANUSCRIPT_THRESHOLD = 0.98
UVIF_REVIEW_MARGIN = 0.05

CONVENTIONAL_NAME = f"Conventional AI ({best_model_name}, tau={CONVENTIONAL_THRESHOLD:.2f})"
UVIF_NAME = f"UVIF ({uvif_model_name}, tau={UVIF_MANUSCRIPT_THRESHOLD:.2f})"

conventional_prob = np.asarray(best_prob).reshape(-1)
conventional_pred = (conventional_prob >= CONVENTIONAL_THRESHOLD).astype(int)

uvif_prob = np.asarray(uvif_prob).reshape(-1)
uvif_pred = (uvif_prob >= UVIF_MANUSCRIPT_THRESHOLD).astype(int)

def diagnostic_summary_row(name, y_true, prob, pred, paradigm):
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    precision = tp / max(tp + fp, 1)
    fn_rate = fn / max(tp + fn, 1)
    fp_rate = fp / max(tn + fp, 1)
    return {
        "Paradigm": paradigm,
        "Model / Decision Rule": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob),
        "FN": int(fn),
        "FP": int(fp),
        "FN rate": fn_rate,
        "FP rate": fp_rate,
    }

# -----------------------------
# Table 1: Quantitative comparison
# -----------------------------
table1_quantitative_comparison = pd.DataFrame([
    diagnostic_summary_row(CONVENTIONAL_NAME, y_test, conventional_prob, conventional_pred, "Conventional AI"),
    diagnostic_summary_row(UVIF_NAME, y_test, uvif_prob, uvif_pred, "UVIF"),
])

table1_quantitative_comparison.to_csv(
    TABLE_DIR / "Table_1_Conventional_AI_vs_UVIF_quantitative_comparison.csv",
    index=False
)
display(table1_quantitative_comparison)

# -----------------------------
# Table 2: Conceptual / decision comparison
# -----------------------------
uvif_lower_review = max(0.0, UVIF_MANUSCRIPT_THRESHOLD - UVIF_REVIEW_MARGIN)
uvif_upper_review = min(1.0, UVIF_MANUSCRIPT_THRESHOLD + UVIF_REVIEW_MARGIN)

uvif_uncertainty = 1.0 - np.abs(uvif_prob - 0.5) * 2.0
uvif_decision_category = np.where(
    uvif_prob >= UVIF_MANUSCRIPT_THRESHOLD,
    "Alert",
    np.where(
        (uvif_prob >= uvif_lower_review) & (uvif_prob < UVIF_MANUSCRIPT_THRESHOLD),
        "Review",
        "No urgent action"
    )
)

conventional_decision_category = np.where(
    conventional_pred == 1,
    "Prediction-only: pneumonia",
    "Prediction-only: normal"
)

case_decision_table = pd.DataFrame({
    "case_id": np.arange(len(y_test)),
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "conventional_probability_pneumonia": conventional_prob,
    "conventional_forced_output": conventional_decision_category,
    "uvif_probability_pneumonia": uvif_prob,
    "uvif_uncertainty": uvif_uncertainty,
    "uvif_decision_category": uvif_decision_category,
})

case_decision_table.to_csv(
    TABLE_DIR / "Table_2_case_level_prediction_only_vs_UVIF_decision_categories.csv",
    index=False
)

decision_summary_rows = []
for category in ["Alert", "Review", "No urgent action"]:
    mask = case_decision_table["uvif_decision_category"] == category
    decision_summary_rows.append({
        "UVIF decision category": category,
        "Meaning": {
            "Alert": "High-risk probability exceeds the conservative UVIF alert threshold.",
            "Review": "Near-threshold evidence is not forced into a class and is routed to review.",
            "No urgent action": "Evidence is below the UVIF alert/review region."
        }[category],
        "Number of cases": int(mask.sum()),
        "Mean pneumonia probability": float(case_decision_table.loc[mask, "uvif_probability_pneumonia"].mean()) if mask.sum() else 0.0,
        "Mean uncertainty": float(case_decision_table.loc[mask, "uvif_uncertainty"].mean()) if mask.sum() else 0.0,
    })

table2_conceptual_decision_comparison = pd.DataFrame([
    {
        "Dimension": "Main objective",
        "Conventional AI": "Minimizes prediction error.",
        "UVIF": "Balances prediction, uncertainty, risk, clinical burden, and safe action.",
    },
    {
        "Dimension": "Output logic",
        "Conventional AI": "Prediction-only output with a forced class decision.",
        "UVIF": "Decision categories: Alert / Review / No urgent action.",
    },
    {
        "Dimension": "Uncertainty handling",
        "Conventional AI": "Uncertainty remains implicit in the probability score.",
        "UVIF": "Uncertainty is explicitly represented in the decision layer.",
    },
    {
        "Dimension": "Risk and burden",
        "Conventional AI": "False positives and false negatives are consequences of a fixed threshold.",
        "UVIF": "False-positive burden, false-negative risk, and review routing are part of the operating rule.",
    },
    {
        "Dimension": "Clinical behavior",
        "Conventional AI": "Every case is classified as normal or pneumonia.",
        "UVIF": "Cases may trigger Alert, Review, or No urgent action depending on equilibrium evidence.",
    },
])

uvif_decision_summary = pd.DataFrame(decision_summary_rows)

table2_conceptual_decision_comparison.to_csv(
    TABLE_DIR / "Table_2_conceptual_prediction_only_vs_UVIF_decision_comparison.csv",
    index=False
)
uvif_decision_summary.to_csv(
    TABLE_DIR / "Table_2_UVIF_decision_category_summary.csv",
    index=False
)

display(table2_conceptual_decision_comparison)
display(uvif_decision_summary)
display(case_decision_table.head(20))

log("\n[TABLE 1] Conventional AI vs UVIF quantitative comparison")
log(table1_quantitative_comparison.to_string(index=False))
log("\n[TABLE 2] Conceptual / decision comparison")
log(table2_conceptual_decision_comparison.to_string(index=False))
log("\n[UVIF DECISION CATEGORY SUMMARY]")
log(uvif_decision_summary.to_string(index=False))


In [ ]:
# ============================================================
# 16E. Figure 1: Conventional AI vs UVIF performance bars
# ============================================================
# Left = Conventional AI, Right = UVIF.
# Metrics: accuracy, balanced accuracy, sensitivity, specificity, F1.

figure1_metrics = ["Accuracy", "Balanced Accuracy", "Sensitivity", "Specificity", "F1"]
figure1_df = table1_quantitative_comparison.set_index("Paradigm").loc[
    ["Conventional AI", "UVIF"], figure1_metrics
]

x = np.arange(len(figure1_metrics))
width = 0.36

plt.figure(figsize=(9.5, 5.0))
plt.bar(x - width/2, figure1_df.loc["Conventional AI"].values, width, label="Conventional AI")
plt.bar(x + width/2, figure1_df.loc["UVIF"].values, width, label="UVIF")
plt.xticks(x, ["Accuracy", "Balanced\naccuracy", "Sensitivity", "Specificity", "F1"])
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.title("Figure 1. Conventional AI versus UVIF performance")
plt.legend()
plt.grid(axis="y", alpha=0.30)
plt.tight_layout()
plt.savefig(FIG_DIR / "Figure_1_Conventional_AI_vs_UVIF_performance_bars.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 16F. Figure 2: Confusion matrices side by side
# ============================================================
# Left: Conventional threshold tau = 0.50.
# Right: UVIF threshold tau = 0.98.

cm_conventional = confusion_matrix(y_test, conventional_pred, labels=[0, 1])
cm_uvif_manuscript = confusion_matrix(y_test, uvif_pred, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))

ConfusionMatrixDisplay(cm_conventional, display_labels=CLASS_NAMES).plot(
    ax=axes[0], values_format="d", colorbar=False
)
axes[0].set_title("Conventional AI\nthreshold tau = 0.50")

ConfusionMatrixDisplay(cm_uvif_manuscript, display_labels=CLASS_NAMES).plot(
    ax=axes[1], values_format="d", colorbar=False
)
axes[1].set_title("UVIF\nthreshold tau = 0.98")

plt.suptitle("Figure 2. Confusion matrices: forced prediction versus UVIF operating rule")
plt.tight_layout()
plt.savefig(FIG_DIR / "Figure_2_Conventional_AI_vs_UVIF_confusion_matrices.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 16G. Figure 3: False-negative / false-positive burden comparison
# ============================================================
# This figure makes the operational burden visible. Lower is better.

burden_counts = table1_quantitative_comparison.set_index("Paradigm").loc[
    ["Conventional AI", "UVIF"], ["FN", "FP"]
]
burden_rates = table1_quantitative_comparison.set_index("Paradigm").loc[
    ["Conventional AI", "UVIF"], ["FN rate", "FP rate"]
]

x = np.arange(2)
width = 0.36

plt.figure(figsize=(8.0, 5.0))
plt.bar(x - width/2, burden_counts["FN"].values, width, label="False negatives")
plt.bar(x + width/2, burden_counts["FP"].values, width, label="False positives")
plt.xticks(x, ["Conventional AI\n(tau=0.50)", "UVIF\n(tau=0.98)"])
plt.ylabel("Number of cases")
plt.title("Figure 3. False-negative and false-positive burden")
plt.legend()
plt.grid(axis="y", alpha=0.30)

for i, paradigm in enumerate(["Conventional AI", "UVIF"]):
    fn_rate = burden_rates.loc[paradigm, "FN rate"]
    fp_rate = burden_rates.loc[paradigm, "FP rate"]
    plt.text(i, max(burden_counts.loc[paradigm]) + 1, f"FN rate={fn_rate:.3f}\nFP rate={fp_rate:.3f}",
             ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(FIG_DIR / "Figure_3_False_negative_false_positive_burden_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 16H. Figure 4: UVIF decision-equilibrium figure
# ============================================================
# Philosophical figure: prediction + uncertainty + risk + review decision.
# This is designed as a manuscript-ready conceptual-quantitative bridge.

decision_counts = case_decision_table["uvif_decision_category"].value_counts()
n_alert = int(decision_counts.get("Alert", 0))
n_review = int(decision_counts.get("Review", 0))
n_no_action = int(decision_counts.get("No urgent action", 0))
n_total = len(case_decision_table)

mean_uncertainty = float(case_decision_table["uvif_uncertainty"].mean())
mean_risk_probability = float(case_decision_table["uvif_probability_pneumonia"].mean())
fp_uvif = int(table1_quantitative_comparison.loc[table1_quantitative_comparison["Paradigm"] == "UVIF", "FP"].iloc[0])
fn_uvif = int(table1_quantitative_comparison.loc[table1_quantitative_comparison["Paradigm"] == "UVIF", "FN"].iloc[0])

fig, ax = plt.subplots(figsize=(12, 6.4))
ax.axis("off")

def box(x, y, text, fontsize=10):
    ax.text(
        x, y, text,
        ha="center", va="center", fontsize=fontsize,
        bbox=dict(boxstyle="round,pad=0.55", linewidth=1.2, facecolor="white")
    )

def arrow(x1, y1, x2, y2):
    ax.annotate(
        "", xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", linewidth=1.4)
    )

box(0.12, 0.62, "Prediction\np(pneumonia)", 10)
box(0.34, 0.78, f"Uncertainty\nmean = {mean_uncertainty:.3f}", 10)
box(0.34, 0.46, f"Risk / burden\nFN = {fn_uvif}, FP = {fp_uvif}", 10)
box(0.57, 0.62, "UVIF equilibrium layer\nprediction + uncertainty + risk + burden", 10)
box(0.82, 0.82, f"Alert\n{n_alert}/{n_total} cases", 10)
box(0.82, 0.62, f"Review\n{n_review}/{n_total} cases", 10)
box(0.82, 0.42, f"No urgent action\n{n_no_action}/{n_total} cases", 10)

arrow(0.20, 0.62, 0.48, 0.62)
arrow(0.39, 0.76, 0.50, 0.66)
arrow(0.39, 0.48, 0.50, 0.58)
arrow(0.66, 0.64, 0.75, 0.80)
arrow(0.66, 0.62, 0.75, 0.62)
arrow(0.66, 0.60, 0.75, 0.44)

ax.text(
    0.5, 0.18,
    "Conventional AI forces a class decision. UVIF converts the same predictive evidence into an operational decision equilibrium.",
    ha="center", va="center", fontsize=11
)
ax.text(
    0.5, 0.08,
    f"UVIF threshold tau = {UVIF_MANUSCRIPT_THRESHOLD:.2f}; review band begins at tau = {uvif_lower_review:.2f}.",
    ha="center", va="center", fontsize=10
)

plt.title("Figure 4. UVIF decision equilibrium: prediction, uncertainty, risk, and safe action", fontsize=14)
plt.tight_layout()
plt.savefig(FIG_DIR / "Figure_4_UVIF_decision_equilibrium.png", dpi=300, bbox_inches="tight")
plt.show()


## Case-level prediction analysis

This section saves representative correct and incorrect cases with their confidence scores.

In [ ]:
# ============================================================
# 17. Case-level correct and incorrect predictions
# ============================================================
case_df = pd.DataFrame({
    "index": np.arange(len(y_test)),
    "true_label": y_test,
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "pred_label": best_pred,
    "pred_class": [CLASS_NAMES[i] for i in best_pred],
    "p_pneumonia": best_prob,
    "confidence": np.maximum(best_prob, 1 - best_prob),
    "outcome": np.where(best_pred == y_test, "Correct", "Error")
}).sort_values(["outcome", "confidence"], ascending=[True, False])

display(case_df.head(20))
case_df.to_csv(TABLE_DIR / "table_case_level_predictions.csv", index=False)

# Show top correct and top incorrect
correct_idx = case_df[case_df["outcome"] == "Correct"].head(4)["index"].tolist()
error_idx = case_df[case_df["outcome"] == "Error"].head(4)["index"].tolist()
selected = correct_idx + error_idx

plt.figure(figsize=(10, 5))
for i, idx in enumerate(selected):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    title = f"T:{CLASS_NAMES[y_test[idx]]}\nP:{CLASS_NAMES[best_pred[idx]]}\nConf:{case_df.loc[case_df['index']==idx, 'confidence'].values[0]:.2f}"
    plt.title(title, fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_case_level_correct_incorrect_predictions.png", dpi=300, bbox_inches="tight")
plt.show()

## Grad-CAM explainability

Grad-CAM is computed using the last convolutional layer of the CNN. This visualizes image regions that most strongly influence the pneumonia prediction.

In [ ]:
# ============================================================
# 18. Grad-CAM utility
# ============================================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="last_conv"):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val
    return heatmap.numpy()

def overlay_heatmap_gray(image, heatmap, alpha=0.45):
    image = image.squeeze()
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], image.shape).numpy().squeeze()
    return image, heatmap_resized

# Select representative pneumonia and normal examples
sample_indices = []
for cls in [0, 1]:
    inds = np.where(y_test == cls)[0]
    # choose high-confidence correct if possible
    correct = [i for i in inds if best_pred[i] == y_test[i]]
    if len(correct) > 0:
        selected_idx = sorted(correct, key=lambda i: np.maximum(best_prob[i], 1-best_prob[i]), reverse=True)[0]
    else:
        selected_idx = inds[0]
    sample_indices.append(selected_idx)

plt.figure(figsize=(8, 4))
for j, idx in enumerate(sample_indices):
    img_array = X_test[idx:idx+1]
    heatmap = make_gradcam_heatmap(img_array, cnn_model, "last_conv")
    image, heatmap_resized = overlay_heatmap_gray(X_test[idx], heatmap)

    plt.subplot(2, 2, 2*j + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Original: {CLASS_NAMES[y_test[idx]]}")
    plt.axis("off")

    plt.subplot(2, 2, 2*j + 2)
    plt.imshow(image, cmap="gray")
    plt.imshow(heatmap_resized, alpha=0.45)
    plt.title(f"Grad-CAM: P(Pneumonia)={cnn_prob[idx]:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "figure_gradcam_representative_cases.png", dpi=300, bbox_inches="tight")
plt.show()

## SHAP explanation on CNN embeddings

For the hybrid model, SHAP is applied to the Random Forest trained on CNN embeddings. These embedding-level explanations help identify which learned representation dimensions influence the pneumonia decision.

In [ ]:
# ============================================================
# 19. SHAP on CNN embeddings -- FIXED robust dimensionality
# ============================================================
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use RandomForest embeddings for stable and fast TreeExplainer analysis.
shap_model_name = "RandomForest Embeddings"
shap_model = trained_ml_models[shap_model_name]

# Limit SHAP sample size for speed.
n_shap = min(300, Z_test.shape[0])
Z_shap = Z_test[:n_shap]

feature_names = [f"emb_{i:03d}" for i in range(Z_shap.shape[1])]

def select_positive_class_shap_values(shap_values, n_samples, n_features):
    """
    Robustly converts SHAP outputs to a 2D array: (samples, features).
    Handles common SHAP formats:
    - list[class] -> each item is (samples, features)
    - ndarray (samples, features)
    - ndarray (samples, features, classes)
    - ndarray (classes, samples, features)
    - Explanation object with .values
    """
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values

    if isinstance(shap_values, list):
        arr = np.asarray(shap_values[1] if len(shap_values) > 1 else shap_values[0])
    else:
        arr = np.asarray(shap_values)

    print("[SHAP] Raw shape:", arr.shape)

    if arr.ndim == 2:
        pass

    elif arr.ndim == 3:
        # Format: (samples, features, classes)
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            cls_idx = 1 if arr.shape[2] > 1 else 0
            arr = arr[:, :, cls_idx]

        # Format: (classes, samples, features)
        elif arr.shape[1] == n_samples and arr.shape[2] == n_features:
            cls_idx = 1 if arr.shape[0] > 1 else 0
            arr = arr[cls_idx, :, :]

        # Less common fallback: squeeze if possible
        else:
            arr = np.squeeze(arr)
            if arr.ndim == 3:
                raise ValueError(f"Unsupported SHAP 3D shape after squeeze: {arr.shape}")

    else:
        arr = np.squeeze(arr)

    if arr.ndim != 2:
        raise ValueError(f"SHAP values must be 2D after correction, got shape {arr.shape}")

    if arr.shape[0] != n_samples and arr.shape[1] == n_samples:
        arr = arr.T

    if arr.shape[1] != n_features:
        raise ValueError(
            f"Feature mismatch: SHAP has {arr.shape[1]} features, "
            f"but Z_shap has {n_features} features."
        )

    print("[SHAP] Corrected shape:", arr.shape)
    return arr

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(Z_shap)

shap_values_to_plot = select_positive_class_shap_values(
    shap_values,
    n_samples=Z_shap.shape[0],
    n_features=Z_shap.shape[1]
)

log("[SHAP] SHAP values generated and corrected.")

shap.summary_plot(
    shap_values_to_plot,
    Z_shap,
    feature_names=feature_names,
    show=False
)
plt.savefig(FIG_DIR / "figure_shap_summary_embeddings.png", dpi=300, bbox_inches="tight")
plt.show()

shap.summary_plot(
    shap_values_to_plot,
    Z_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.savefig(FIG_DIR / "figure_shap_bar_embeddings.png", dpi=300, bbox_inches="tight")
plt.show()

# Save mean absolute SHAP importance.
mean_abs_shap = np.abs(shap_values_to_plot).mean(axis=0).reshape(-1)

if len(feature_names) != len(mean_abs_shap):
    log(
        f"[SHAP] Feature-name mismatch: {len(feature_names)} names vs "
        f"{len(mean_abs_shap)} SHAP values. Using generic names."
    )
    feature_names_fixed = [f"emb_{i:03d}" for i in range(len(mean_abs_shap))]
else:
    feature_names_fixed = feature_names

shap_importance_df = pd.DataFrame({
    "embedding_feature": feature_names_fixed,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance_df.head(20))
shap_importance_df.to_csv(TABLE_DIR / "table_shap_embedding_importance.csv", index=False)

log("[SHAP] Embedding-level importance table saved.")

## Optional LIME image explanation

This cell uses LIME for image explanations. It may take a few minutes. It is useful for case-level interpretation, but Grad-CAM is usually more stable for CNN-based medical imaging.

In [ ]:
# ============================================================
# 20. Optional LIME image explanation
# ============================================================
from lime import lime_image
from skimage.segmentation import mark_boundaries

def cnn_predict_for_lime(images):
    # LIME sends RGB-like images; convert to grayscale if needed
    arr = np.array(images)
    if arr.ndim == 4 and arr.shape[-1] == 3:
        arr = arr.mean(axis=-1, keepdims=True)
    elif arr.ndim == 3:
        arr = arr[..., np.newaxis]
    arr = arr.astype("float32")
    if arr.max() > 1:
        arr = arr / 255.0
    p = cnn_model.predict(arr, verbose=0).reshape(-1)
    return np.column_stack([1 - p, p])

try:
    lime_idx = sample_indices[1] if len(sample_indices) > 1 else 0
    lime_img = X_test[lime_idx].squeeze()
    # LIME expects 3 channels for image segmentation; repeat grayscale channels
    lime_img_rgb = np.repeat(lime_img[..., np.newaxis], 3, axis=-1)

    explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)
    explanation = explainer_lime.explain_instance(
        lime_img_rgb,
        cnn_predict_for_lime,
        top_labels=2,
        hide_color=0,
        num_samples=500
    )

    temp, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(mark_boundaries(temp, mask))
    plt.title("LIME Image Explanation - Pneumonia Class")
    plt.axis("off")
    plt.savefig(FIG_DIR / "figure_lime_image_explanation.png", dpi=300, bbox_inches="tight")
    plt.show()

except Exception as e:
    log(f"[WARNING] LIME image explanation failed: {e}")

In [ ]:
# ============================================================
# 21. Save models and final summary
# ============================================================
cnn_model.save(MODEL_DIR / "compact_cnn_pneumoniamnist.keras")
embedding_model.save(MODEL_DIR / "cnn_embedding_extractor.keras")
joblib.dump(trained_ml_models, MODEL_DIR / "embedding_ml_models.joblib")
joblib.dump(soft_voting, MODEL_DIR / "soft_voting_embeddings.joblib")

summary = {
    "dataset_path": DATA_PATH,
    "output_directory": str(BASE_DIR),
    "best_model_by_f1": best_model_name,
    "uvif_selected_model": globals().get("uvif_model_name", None),
    "uvif_selected_threshold": globals().get("uvif_tau_star", None),
    "cnn_result": cnn_result,
    "all_results": all_results_df.to_dict(orient="records"),
    "table1_conventional_vs_uvif": globals().get("table1_quantitative_comparison", pd.DataFrame()).to_dict(orient="records"),
    "table2_conceptual_decision_comparison": globals().get("table2_conceptual_decision_comparison", pd.DataFrame()).to_dict(orient="records"),
    "uvif_decision_summary": globals().get("uvif_decision_summary", pd.DataFrame()).to_dict(orient="records"),
    "requested_manuscript_artifacts": {
        "tables": [
            "Table_1_Conventional_AI_vs_UVIF_quantitative_comparison.csv",
            "Table_2_conceptual_prediction_only_vs_UVIF_decision_comparison.csv",
            "Table_2_case_level_prediction_only_vs_UVIF_decision_categories.csv",
            "Table_2_UVIF_decision_category_summary.csv"
        ],
        "figures": [
            "Figure_1_Conventional_AI_vs_UVIF_performance_bars.png",
            "Figure_2_Conventional_AI_vs_UVIF_confusion_matrices.png",
            "Figure_3_False_negative_false_positive_burden_comparison.png",
            "Figure_4_UVIF_decision_equilibrium.png"
        ]
    }
}

with open(OTHER_DIR / "experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

log("\n[FINAL SUMMARY]")
log(f"Best model by F1: {best_model_name}")
if "uvif_model_name" in globals():
    log(f"UVIF-selected Diagnostics model: {uvif_model_name}")
    log(f"UVIF-selected Diagnostics threshold: {uvif_tau_star:.2f}")
log(f"Outputs saved to: {BASE_DIR}")
log("Manuscript-ready comparison artifacts generated: 2 tables + 4 figures.")
log("Notebook completed successfully.")
print(f"Done. Outputs saved to: {BASE_DIR}")


## Suggested manuscript positioning

This notebook now supports a stronger Diagnostics-oriented paper framing such as:

**UVIF-Guided Explainable and Calibration-Aware AI for High-Sensitivity Pneumonia Screening: A Trust-Aware Diagnostic Decision-Support Framework Using PneumoniaMNIST**

Recommended methodological contribution:
- A compact CNN is used to learn image representations from PneumoniaMNIST.
- Classical ensemble learning models are trained on the learned embeddings.
- A restrained UVIF layer supports trust-aware model selection and threshold control.
- Diagnostics-oriented curves report ROC, calibration, sensitivity-specificity, precision-recall, false-negative/false-positive burden, and UVIF threshold utility.
- Grad-CAM and LIME explain image-level attention.
- SHAP explains embedding-level decision factors.
- Calibration and case-level analysis support trust-aware interpretation.

Recommended claim boundary:
- The framework should be described as screening-oriented diagnostic decision support, not as a clinically deployable autonomous diagnostic system.
- UVIF should be presented as a practical reliability-and-decision layer rather than as a replacement for clinical validation.
